In [8]:
!pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [9]:
from langchain_core.documents import Document

In [10]:
sample_data=Document(
    page_content="Hello Rudra",
    metadata={'source':"https://google.com"}
)

In [11]:
print(type(sample_data))

<class 'langchain_core.documents.base.Document'>


In [12]:
from langchain_community.document_loaders.text import TextLoader

Ingestion pipeline

In [13]:
import os
from langchain_community.document_loaders import PyPDFLoader

In [15]:
import zipfile
import os
zip_file_path = '/content/data.zip'
extract_dir = '/content/data'

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"'{zip_file_path}' unzipped to '{extract_dir}'")


'/content/data.zip' unzipped to '/content/data'


In [26]:
from langchain_community.document_loaders import PyPDFLoader
loader=PyPDFLoader('git-cheat-sheet-education.pdf')
doc=loader.load()
doc

[Document(metadata={'producer': 'Mac OS X 10.9.1 Quartz PDFContext', 'creator': 'Adobe Illustrator CC (Macintosh)', 'creationdate': "D:20140224195805Z00'00'", 'title': 'git-cheat-sheet-education', 'moddate': "D:20140224195805Z00'00'", 'source': 'git-cheat-sheet-education.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content="GIT CHEAT SHEET\nSTAGE & SNAPSHOT\nWorking with snapshots and the Git staging area\ngit status\nshow modiﬁed ﬁles in working directory, staged for your next commit\ngit add [file]\nadd a ﬁle as it looks now to your next commit (stage)\ngit reset [file]\nunstage a ﬁle while retaining the changes in working directory\ngit diff\ndiﬀ of what is changed but not staged\ngit diff --staged\ndiﬀ of what is staged but not yet committed\ngit commit -m “[descriptive message]”\ncommit your staged content as a new commit snapshot\nSETUP\nConﬁguring user information used across all local repositories\ngit config --global user.name “[firstname lastname]”\nset a name 

In [27]:
def all_file_load():
  folder_path='data/data'
  num_doc=0
  files=[]
  for file in os.listdir(folder_path):
    if file.lower().endswith('.pdf'):
      pdf_path=os.path.join(folder_path,file)
      loader=PyPDFLoader(pdf_path)
      doc=loader.load()
      files.extend(doc)
      num_doc+=1

  print('toatal files ',num_doc)
  print('total pages=',len(files))
  return files

In [28]:
all_pdf_files=all_file_load()

toatal files  8
total pages= 69


In [29]:
pip install langchain-text-splitters

Convert the Document into chunks


In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
def chunk_form(document,chunk_size=500,chunk_overlap=50):
  text_splitter=RecursiveCharacterTextSplitter(
      chunk_size=chunk_size,
      chunk_overlap=chunk_overlap
  )
  chunk_docs=text_splitter.split_documents(document)
  return chunk_docs

In [20]:
chunks=chunk_form(all_pdf_files)

In [30]:
len(chunks)

349

In [31]:
chunks

[Document(metadata={'producer': 'PDPreStamp v3.3', 'creator': 'Adobe InDesign 16.4 (Windows)', 'creationdate': '2022-09-29T17:54:00+02:00', 'author': 'ISO', 'license': 'Information Handling Services, 2022', 'moddate': '2022-10-26T19:22:40+08:00', 'title': 'ISO/IEC 27001:2022', 'trapped': '/False', 'source': 'data/data/iso27001.pdf', 'total_pages': 26, 'page': 0, 'page_label': '1'}, page_content="Information security, cybersecurity \nand privacy protection — Information \nsecurity management systems — \nRequirements\nSécurité de l'information, cybersécurité et protection de la vie \nprivée — Systèmes de management de la sécurité de l'information — \nExigences\nINTERNATIONAL \nSTANDARD\nISO/IEC \n27001\nThird edition  \n2022-10\nReference number \nISO/IEC 27001:2022(E)\n© ISO/IEC 2022\n--``,,,,,``````,,,,,`,`,`,`,,`,-`-`,,`,,`,`,,`---"),
 Document(metadata={'producer': 'PDPreStamp v3.3', 'creator': 'Adobe InDesign 16.4 (Windows)', 'creationdate': '2022-09-29T17:54:00+02:00', 'author': 'I

Chunk Embeding


In [32]:
from sentence_transformers import SentenceTransformer

In [67]:
class Embeding:
  def __init__(self,model_name='all-MiniLM-L6-v2'):
    self.model_name=model_name
    print('model loading......')
    self.model=SentenceTransformer(self.model_name)
    print('Embedding Dimension =', self.model.get_sentence_embedding_dimension())
  def generate_embedding(self,text):
    embede_text=self.model.encode(text,show_progress_bar=True)
    print('embedding shape=',embede_text.shape)
    return embede_text.tolist()

In [68]:
embedding=Embeding()


model loading......


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding Dimension = 384


Vector Store


In [69]:
import chromadb
import uuid #help to create index

In [75]:
from langchain_core import documents
class VectorStore:
  def __init__(self,persist_directory='data/vector_Store',collection_name='pdf_doc'):
    self.persist_directory=persist_directory
    self.collection_name=collection_name
    self.collection=None
    self.client=None
    self.initialize_store()
  def initialize_store(self):
    os.makedirs(self.persist_directory,exist_ok=True)
    #create clint
    self.client=chromadb.PersistentClient(path=self.persist_directory)

    #collection
    self.collection=self.client.get_or_create_collection(
        name=self.collection_name,
        metadata={'desc':'This is a vector Store Collection for pdf->chumk emeding by RAG'}

    )
    print('initialize the Vectore store ',self.collection_name)
    print('total_collection count=',self.collection.count())
  def add_documents(self,documents,embeddings):
    if len(documents) !=len(embeddings):
      raise ValueError('Size mismatch')
    ids=[]
    meta_datas=[]
    doc_lists=[]
    emding_list=[]

    for i,(doc,embedding) in enumerate(zip(documents,embeddings)):
      doc_id=f"doc_{uuid.uuid4()}"
      ids.append(doc_id)

      metadata=dict(doc.metadata)
      metadata['idx']=i
      meta_datas.append(metadata)

      doc_lists.append(doc.page_content)
      emding_list.append(embedding)
      self.collection.add(
          ids=ids,
          metadatas=meta_datas,
          documents=doc_lists,
          embeddings=emding_list

      )
    print('total added=',len(doc_lists))






In [76]:
doc=uuid.uuid4()
doc

UUID('45e341bb-259c-4822-93d4-12d768d72540')

In [77]:
vector_store=VectorStore()

initialize the Vectore store  pdf_doc
total_collection count= 0


In [78]:
texts=[doc.page_content for doc in chunks]
embedder = Embeding()
embedding = embedder.generate_embedding(texts)
vector_store.add_documents(chunks,embedding)

model loading......


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding Dimension = 384


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

embedding shape= (349, 384)
total added= 349


Retriver Pipeline

In [79]:
from sklearn.metrics.pairwise import cosine_similarity

In [89]:
class RAgRetriver:
    def __init__(self, Embeding, vector_store):
        self.embedding_model = Embeding()
        self.vector_store = vector_store

    def retriver(self, query, top_k=5, score_thereshold=0.0):
        # Generate embedding (input must be list)
        query_embedding = self.embedding_model.generate_embedding([query])[0]

        # Query vector DB (Chroma)
        result = self.vector_store.collection.query(
            query_embeddings=[query_embedding],
            n_results=top_k
        )

        retrieved_docs = []

        # Check results safely
        if result.get('documents') and len(result['documents']) > 0:

            ids = result['ids'][0]
            metadatas_list = result['metadatas'][0]
            documents_list = result['documents'][0]
            distances_list = result['distances'][0]

            for i, (doc_id, doc, metadata, distance) in enumerate(
                zip(ids, documents_list, metadatas_list, distances_list)
            ):
                similarity_score = 1 - distance

                if similarity_score >= score_thereshold:
                    retrieved_docs.append({
                        'id': doc_id,
                        'document': doc,
                        'metadata': metadata,
                        'distance': distance,
                        'similarity_score': similarity_score,
                        'rank': i + 1
                    })

        if retrieved_docs:
            print("retrieved documents =", len(retrieved_docs))
            return retrieved_docs
        else:
            print("Oops no Document Found")
            return []

In [90]:
rag_retriver=RAgRetriver(Embeding,vector_store)

model loading......


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding Dimension = 384


In [95]:
rag_retriver.retriver('What is Ethics')

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape= (1, 384)
retrieved documents = 5


[{'id': 'doc_4b67f875-e36b-455a-9e41-a43b46e7b0e1',
  'document': 'deeply ingrained and influence an individual’s personal decisions and behaviour. \nWhat are Ethics? \nEthics are a set of rules or guidelines established by a group, profession, or society to govern \nacceptable behaviour. Unlike morals, which are personal, ethics are often codified into \nprofessional codes of conduct, workplace policies, and even laws. \n• A lawyer may have to defend a client, even if they personally believe the client is guilty.',
  'metadata': {'page': 0,
   'creator': 'Microsoft® Word 2021',
   'moddate': '2025-04-01T20:19:08+05:30',
   'page_label': '1',
   'total_pages': 7,
   'creationdate': '2025-04-01T20:19:08+05:30',
   'author': 'Amlan Asutosh',
   'source': 'data/data/Unit 4.pdf',
   'producer': 'Microsoft® Word 2021',
   'idx': 248},
  'distance': 0.48669907450675964,
  'similarity_score': 0.5133009254932404,
  'rank': 1},
 {'id': 'doc_b3cc80c4-b771-4fda-9ca4-76a11cceb151',
  'document': '